# Music Genre Classification with MERT-v1-330M on GTZAN

Fine-tunes [MERT-v1-330M](https://huggingface.co/m-a-p/MERT-v1-330M) for **music genre classification** on [GTZAN](https://huggingface.co/datasets/marsyas/gtzan).



In [2]:
!pip install -q --upgrade transformers datasets evaluate accelerate huggingface_hub gradio librosa soundfile nnAudio

!nvidia-smi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.0 MB/s eta 0:00:00
Fri Jul 24 09:33:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                              

In [ ]:
# Update to required versions, then hard-restart the kernel to clear any memory lock
!pip install -q -U datasets pyarrow fsspec

import os
os.kill(os.getpid(), 9)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.4.0 which is incompatible.


> After the kernel restarts, **re-run from the cell below** (no need to reinstall — that's already done).

## 1. Loading the Dataset

In [1]:
from datasets import load_dataset

gtzan = load_dataset("confit/gtzan-parquet")
print(gtzan)
print(gtzan["train"][0].keys())

README.md:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

data/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  294MB            

data/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  293MB            

data/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/443 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/197 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/290 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['audio', 'genre', 'label'],
        num_rows: 443
    })
    validation: Dataset({
        features: ['audio', 'genre', 'label'],
        num_rows: 197
    })
    test: Dataset({
        features: ['audio', 'genre', 'label'],
        num_rows: 290
    })
})
dict_keys(['audio', 'genre', 'label'])


## 2. Genre Label Mapping

`confit/gtzan-parquet` stores `genre` as a plain string column (not a `ClassLabel`), so we build the
`id2label` / `label2id` maps ourselves from the unique values, rather than relying on
`features["genre"].names` (which will raise `AttributeError` on a plain string column).

In [2]:
unique_genres = sorted(set(gtzan["train"]["genre"]))
label2id = {genre: i for i, genre in enumerate(unique_genres)}
id2label = {i: genre for i, genre in enumerate(unique_genres)}

print("Genres:", unique_genres)

Genres: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']


## 3. Train / Test Split

GTZAN only ships a single split, so we carve out a held-out test set ourselves.

In [3]:
from datasets import ClassLabel

gtzan["train"] = gtzan["train"].cast_column("genre", ClassLabel(names=unique_genres))
gtzan = gtzan["train"].train_test_split(test_size=0.1, seed=42, stratify_by_column="genre")
print(gtzan)

Casting the dataset:   0%|          | 0/443 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['audio', 'genre', 'label'],
        num_rows: 398
    })
    test: Dataset({
        features: ['audio', 'genre', 'label'],
        num_rows: 45
    })
})


## 4. (Optional) Listen to a Few Samples

Not part of training — safe to skip. `debug=False` and no `share=True` so it doesn't hang the notebook
or spin up a public tunnel; this cell can simply be skipped entirely for the final submission run.

In [4]:
import gradio as gr

def generate_audio():
    example = gtzan["train"].shuffle()[0]
    audio = example["audio"]
    return (audio["sampling_rate"], audio["array"]), example["genre"]

with gr.Blocks() as demo:
    with gr.Column():
        for _ in range(4):
            audio, label = generate_audio()
            gr.Audio(audio, label=label)

demo.launch(debug=False, share=False)

/usr/local/lib/python3.12/dist-packages/gradio/processing_utils.py:724: UserWarning: Trying to convert audio automatically from float32 to 16-bit int format.
  warnings.warn(warning.format(data.dtype))
/usr/local/lib/python3.12/dist-packages/gradio/processing_utils.py:724: UserWarning: Trying to convert audio automatically from float32 to 16-bit int format.
  warnings.warn(warning.format(data.dtype))
/usr/local/lib/python3.12/dist-packages/gradio/processing_utils.py:724: UserWarning: Trying to convert audio automatically from float32 to 16-bit int format.
  warnings.warn(warning.format(data.dtype))
/usr/local/lib/python3.12/dist-packages/gradio/processing_utils.py:724: UserWarning: Trying to convert audio automatically from float32 to 16-bit int format.
  warnings.warn(warning.format(data.dtype))


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

## 5. Feature Extraction with MERT

MERT ships its own `AutoFeatureExtractor`. Note `trust_remote_code=True` — MERT relies on custom code
(and the `nnAudio` package installed above) for its CQT-based feature pipeline.

In [5]:
from transformers import AutoFeatureExtractor

model_id = "m-a-p/MERT-v1-330M"
feature_extractor = AutoFeatureExtractor.from_pretrained(
    model_id, trust_remote_code=True, do_normalize=True, return_attention_mask=True
)
sampling_rate = feature_extractor.sampling_rate
print("Target sampling rate:", sampling_rate)

preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

Target sampling rate: 24000


In [6]:
from datasets import Audio

gtzan = gtzan.cast_column("audio", Audio(sampling_rate=sampling_rate))

## 6. Preprocessing

Every clip is capped at 30s. Crucially, we **pad to a fixed max length** (`padding="max_length"`) so
every example in a batch has the same tensor shape — without this, the default data collator has
nothing consistent to stack and training crashes on the first batch. We also convert the `genre`
string straight into an integer `label` column here, so nothing downstream depends on a column that
doesn't exist.

In [7]:
max_duration = 30.0

def preprocess_function(examples):
    audio_arrays = [x["array"] for x in examples["audio"]]
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=feature_extractor.sampling_rate,
        max_length=int(feature_extractor.sampling_rate * max_duration),
        truncation=True,
        padding="max_length",
        return_attention_mask=True,
    )
    inputs["label"] = examples["genre"]
    return inputs

columns_to_drop = [c for c in gtzan["train"].column_names if c not in ("input_values", "attention_mask", "label")]

gtzan_encoded = gtzan.map(
    preprocess_function,
    remove_columns=columns_to_drop,
    batched=True,
    batch_size=100,
    num_proc=1,
)
gtzan_encoded.set_format(type="torch", columns=["input_values", "attention_mask", "label"])
print(gtzan_encoded)
print(gtzan_encoded["train"][0].keys())

Map (num_proc=1):   0%|          | 0/398 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/45 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_values', 'attention_mask'],
        num_rows: 398
    })
    test: Dataset({
        features: ['label', 'input_values', 'attention_mask'],
        num_rows: 45
    })
})
dict_keys(['label', 'input_values', 'attention_mask'])


## 7. Model Architecture

MERT is a self-supervised backbone, not a classifier out of the box, so we wrap it in a custom
`nn.Module`. This is the **one, consolidated** version of the model class — the earlier draft
redefined `MERTForAudioClassification` three times with different (incompatible) signatures, and the
last one in execution order silently won overwriting the others.

**Partial fine-tuning:** freeze the whole MERT backbone, then unfreeze only the last
`num_unfrozen_layers` transformer layers. This cuts trainable parameters and optimizer memory a lot,
which matters on a single T4.

In [8]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoConfig
from transformers.modeling_outputs import SequenceClassifierOutput

class MERTForAudioClassification(nn.Module):
    def __init__(self, model_id, num_labels, id2label, label2id, num_unfrozen_layers=4):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
        self.mert = AutoModel.from_pretrained(model_id, trust_remote_code=True)

        hidden_size = self.config.hidden_size
        self.classifier = nn.Linear(hidden_size, num_labels)

        self.num_labels = num_labels
        self.id2label = id2label
        self.label2id = label2id

        for param in self.mert.parameters():
            param.requires_grad = False

        encoder = self.mert.encoder
        if hasattr(encoder, "layers"):
            transformer_layers = encoder.layers
        elif hasattr(encoder, "layer"):
            transformer_layers = encoder.layer
        else:
            transformer_layers = []

        for layer in transformer_layers[-num_unfrozen_layers:]:
            for param in layer.parameters():
                param.requires_grad = True

    def forward(self, input_values, attention_mask=None, labels=None, **kwargs):
        outputs = self.mert(input_values, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state  # (batch, frames, hidden) -- frame-level, not sample-level

        if attention_mask is not None and hasattr(self.mert, "_get_feature_vector_attention_mask"):
            # The conv feature encoder downsamples raw audio into frames (e.g. 720,000 samples -> ~2,249
            # frames), so the raw-waveform attention_mask has to be downsampled to match before pooling.
            feature_attention_mask = self.mert._get_feature_vector_attention_mask(
                hidden_states.shape[1], attention_mask
            )
            mask = feature_attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
            sum_embeddings = torch.sum(hidden_states * mask, 1)
            sum_mask = torch.clamp(mask.sum(1), min=1e-9)
            pooled_output = sum_embeddings / sum_mask
        else:
            pooled_output = torch.mean(hidden_states, dim=1)

        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        return SequenceClassifierOutput(loss=loss, logits=logits)


num_labels = len(id2label)
num_unfrozen_layers = 4

model = MERTForAudioClassification(
    model_id=model_id,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    num_unfrozen_layers=num_unfrozen_layers,
)

config.json:   0%|          | 0.00/2.03k [00:00<?, ?B/s]

configuration_MERT.py:   0%|          | 0.00/5.34k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/m-a-p/MERT-v1-330M:
- configuration_MERT.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_MERT.py:   0%|          | 0.00/18.0k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/m-a-p/MERT-v1-330M:
- modeling_MERT.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.26GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/403 [00:00<?, ?it/s]

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Using device: {device}")

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")

Using device: cuda
Trainable parameters: 50,395,146 / 315,439,242


## 8. Hugging Face Hub Login

In [10]:
from huggingface_hub import notebook_login

notebook_login()

## 9. Training Arguments

Small batch size with gradient accumulation, since 30s audio clips through a 330M-parameter backbone
are memory-hungry even with most of the backbone frozen.

In [12]:
from transformers import TrainingArguments

model_name = model_id.split("/")[-1]

training_args = TrainingArguments(
    output_dir=f"{model_name}-finetuned-gtzan",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=10,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
    push_to_hub=True,
    report_to="none",
)

## 10. Evaluation Metric

In [13]:
import evaluate
import numpy as np

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

## 11. Trainer Setup and Training

One `Trainer` instantiation, one `.train()` call — the previous draft had the model, dataset-prep
logic, and `trainer.train()` each duplicated two or three times in a row, which is what made it hard
to tell which version of anything was actually running.

In [21]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=gtzan_encoded["train"],
    eval_dataset=gtzan_encoded["test"],
    compute_metrics=compute_metrics
)

trainer.train()

SyntaxError: positional argument follows keyword argument (3178999128.py, line 9)

## 12. Sharing the Model on the Hub

In [15]:
kwargs = {
    "dataset_tags": ["marsyas/gtzan"],
    "dataset": "GTZAN",
    "model_name": f"{model_name}-finetuned-gtzan",
    "finetuned_from": model_id,
    "tasks": "audio-classification",
}

trainer.push_to_hub(**kwargs)

CommitInfo(commit_url='https://huggingface.co/Atrac/MERT-v1-330M-finetuned-gtzan/commit/aff6b257f9b1cf05eff7da3cb87e27433fef7d7d', commit_message='End of training', commit_description='', oid='aff6b257f9b1cf05eff7da3cb87e27433fef7d7d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Atrac/MERT-v1-330M-finetuned-gtzan', endpoint='https://huggingface.co', repo_type='model', repo_id='Atrac/MERT-v1-330M-finetuned-gtzan'), pr_revision=None, pr_num=None)

CommitInfo(commit_url='https://huggingface.co/Atrac/MERT-v1-330M-finetuned-gtzan/commit/aff6b257f9b1cf05eff7da3cb87e27433fef7d7d', commit_message='End of training', commit_description='', oid='aff6b257f9b1cf05eff7da3cb87e27433fef7d7d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Atrac/MERT-v1-330M-finetuned-gtzan', endpoint='https://huggingface.co', repo_type='model', repo_id='Atrac/MERT-v1-330M-finetuned-gtzan'), pr_revision=None, pr_num=None)

## 13. Inference

In [16]:
from transformers import pipeline

# 1. Dynamically add the missing device property to the model object
trainer.model.device = next(trainer.model.parameters()).device

# 2. Build the pipeline normally
pipe = pipeline(
    "audio-classification",
    model=trainer.model,
    feature_extractor=feature_extractor
)

[transformers] The model 'MERTForAudioClassification' is not supported for audio-classification. Supported models are ['ASTForAudioClassification', 'Data2VecAudioForSequenceClassification', 'HubertForSequenceClassification', 'SEWForSequenceClassification', 'SEWDForSequenceClassification', 'UniSpeechForSequenceClassification', 'UniSpeechSatForSequenceClassification', 'Wav2Vec2ForSequenceClassification', 'Wav2Vec2BertForSequenceClassification', 'Wav2Vec2ConformerForSequenceClassification', 'WavLMForSequenceClassification', 'WhisperForAudioClassification'].
[transformers] The model 'MERTForAudioClassification' is not supported for audio-classification. Supported models are ['ASTForAudioClassification', 'Data2VecAudioForSequenceClassification', 'HubertForSequenceClassification', 'SEWForSequenceClassification', 'SEWDForSequenceClassification', 'UniSpeechForSequenceClassification', 'UniSpeechSatForSequenceClassification', 'Wav2Vec2ForSequenceClassification', 'Wav2Vec2BertForSequenceClassific